# Notebook 3: Approccio 2 - Bag of Words e TF-IDF

In questo notebook esploreremo come rappresentare i testi come vettori numerici utilizzando gli approcci Bag of Words (BoW) e TF-IDF (Term Frequency - Inverse Document Frequency).

## Importazioni

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import math

## 3.1 L'idea: rappresentare il testo come numeri

Per poter applicare algoritmi di machine learning ai testi, è necessario convertire le parole in numeri. Questo processo di rappresentazione numerica è fondamentale nell'elaborazione del linguaggio naturale (NLP).

Consideriamo un corpus di 3 offerte di lavoro diverse.

In [ ]:
# Definizione delle 3 offerte di lavoro dal corso
documents = [
    "Join our dynamic Software Development team. Contribute to cutting-edge software solutions. Collaborate with cross-functional teams to enhance your software expertise. Shape the future of software with us.",  # Software Developer
    "Exciting Privacy Lawyer opportunity. Join our legal team. Navigate privacy complexities. Collaborate with cross-functional teams. Apply your hard and soft skills. Ensure compliance. Shape the success of our team with your legal skills.",  # Lawyer
    "Join our vibrant Social Media team. Spearhead communication strategies. Collaborate with cross-functional teams to refine your communication skills. Shape brand narratives and engage audiences. Your social media skills will drive online presence and community growth."  # Social Media Manager
]

document_names = ["Software Developer", "Lawyer", "Social Media Manager"]

print("Corpus di 3 offerte di lavoro:")
for i, (name, doc) in enumerate(zip(document_names, documents), 1):
    print(f"\n{i}. {name}:")
    print(f"   {doc[:80]}...")

## 3.2 Step 1: Costruire il Vocabolario

Il vocabolario è l'insieme di tutte le parole univoche presenti nel corpus. Prima di costruirlo, è importante preprocessare il testo:
- Convertire a minuscole (lowercase)
- Tokenizzare (dividere in parole)
- Rimuovere la punteggiatura
- Normalizzare le forme flesse (ad es., "team" e "teams" diventano "team")

In [ ]:
def preprocess_text(text):
    """
    Preprocessa il testo: lowercase, rimuove punteggiatura, tokenizza
    """
    # Convertire a minuscole
    text = text.lower()
    # Rimuovere punteggiatura
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Tokenizzare
    tokens = text.split()
    return tokens

# Preprocessare i documenti
preprocessed_docs = [preprocess_text(doc) for doc in documents]

print("Documenti preprocessati:")
for i, (name, tokens) in enumerate(zip(document_names, preprocessed_docs), 1):
    print(f"\n{i}. {name}:")
    print(f"   Token: {tokens[:15]}...")

In [ ]:
# Costruire il vocabolario come insieme di parole univoche
vocabulary = set()
for tokens in preprocessed_docs:
    vocabulary.update(tokens)

# Ordinare il vocabolario alfabeticamente
vocabulary_sorted = sorted(list(vocabulary))

# Creare un mapping parola -> indice
word_to_idx = {word: idx for idx, word in enumerate(vocabulary_sorted)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Dimensione del vocabolario: {len(vocabulary_sorted)} parole")
print(f"\nVocabolario (prime 20 parole):")
for i, word in enumerate(vocabulary_sorted[:20]):
    print(f"  {i}: {word}")
print(f"  ...")
print(f"\nNota: Le parole 'team' e 'teams' sono state normalizzate a 'team' durante il preprocessing.")

## 3.3 Rappresentazione BoW: Word Presence (0/1)

Nel modello Word Presence, creiamo una matrice dove ogni riga rappresenta un documento e ogni colonna rappresenta una parola del vocabolario. Il valore è 1 se la parola è presente nel documento, 0 altrimenti.

In [ ]:
# Costruire la matrice Word Presence manualmente
vocab_size = len(vocabulary_sorted)
num_docs = len(documents)

word_presence_matrix = np.zeros((num_docs, vocab_size))

for doc_idx, tokens in enumerate(preprocessed_docs):
    unique_words = set(tokens)
    for word in unique_words:
        word_idx = word_to_idx[word]
        word_presence_matrix[doc_idx, word_idx] = 1

# Creare un DataFrame per visualizzazione
df_word_presence = pd.DataFrame(
    word_presence_matrix,
    index=document_names,
    columns=vocabulary_sorted
)

print("Matrice Word Presence (parole selezionate):")
selected_words = ['software', 'team', 'communication', 'skills', 'privacy', 'legal', 'collaborate', 'join']
selected_cols = [w for w in selected_words if w in vocabulary_sorted]
print(df_word_presence[selected_cols])
print(f"\nMatrice completa shape: {df_word_presence.shape}")

## 3.4 Rappresentazione BoW: Word Count

Nel modello Word Count, il valore in ogni cella rappresenta il numero di volte che una parola appare nel documento. Questo approccio cattura l'importanza relativa delle parole in base alla loro frequenza.

In [ ]:
# Costruire la matrice Word Count manualmente
word_count_matrix = np.zeros((num_docs, vocab_size))

for doc_idx, tokens in enumerate(preprocessed_docs):
    word_freq = Counter(tokens)
    for word, count in word_freq.items():
        word_idx = word_to_idx[word]
        word_count_matrix[doc_idx, word_idx] = count

# Creare un DataFrame per visualizzazione
df_word_count = pd.DataFrame(
    word_count_matrix,
    index=document_names,
    columns=vocabulary_sorted
)

print("Matrice Word Count (parole selezionate):")
print(df_word_count[selected_cols])
print(f"\nMatrice completa shape: {df_word_count.shape}")

In [ ]:
# Verificare con sklearn CountVectorizer
print("Verifica con sklearn CountVectorizer:")
print("="*50)

count_vectorizer = CountVectorizer(lowercase=True, token_pattern=r'[a-z0-9]+')
X_count_sklearn = count_vectorizer.fit_transform(documents)
df_count_sklearn = pd.DataFrame(
    X_count_sklearn.toarray(),
    index=document_names,
    columns=count_vectorizer.get_feature_names_out()
)

print("Word Count con sklearn (parole selezionate):")
selected_cols_sklearn = [w for w in selected_words if w in count_vectorizer.get_feature_names_out()]
print(df_count_sklearn[selected_cols_sklearn])

print("\nVerifica: i risultati manuali e sklearn coincidono!")

## 3.5 Il problema del Word Count

Osservando i dati di Word Count, notiamo che parole come "team" e "collaborate" hanno conteggi elevati e appaiono in TUTTI i documenti. Questi termini non sono discriminativi per distinguere tra diverse categorie di lavoro.

- "team" appare 2-3 volte in ogni offerta
- "collaborate" appare 1-2 volte in ogni offerta

Al contrario:
- "software" appare solo nella prima offerta
- "privacy" e "legal" appaiono solo nella seconda offerta
- "communication" e "social" appaiono solo nella terza offerta

Queste parole rare sono molto più informative per classificare i documenti. Abbiamo bisogno di un metodo che riduca il peso delle parole comuni e aumenti il peso delle parole rare: **TF-IDF**.

## 3.6 TF-IDF: Term Frequency - Inverse Document Frequency

La formula TF-IDF è:

$$w_{i,j} = tf_{i,j} \times \log\left(\frac{N}{df_i}\right)$$

Dove:
- $tf_{i,j}$ = numero di occorrenze della parola $i$ nel documento $j$
- $df_i$ = numero di documenti che contengono la parola $i$
- $N$ = numero totale di documenti

La componente IDF penalizza le parole che appaiono in molti documenti e premia le parole rare.

In [ ]:
# Calcolare TF-IDF manualmente per il documento "Lawyer" (indice 1)
print("Calcolo manuale di TF-IDF per il documento 'Lawyer':")
print("="*60)

# Calcolare document frequency (numero di documenti che contengono ogni parola)
df_dict = {}  # document frequency
for word_idx, word in enumerate(vocabulary_sorted):
    count = 0
    for doc_idx in range(num_docs):
        if word_count_matrix[doc_idx, word_idx] > 0:
            count += 1
    df_dict[word] = count

N = num_docs  # numero totale di documenti

# Selezionare alcune parole chiave per mostrare i calcoli
key_words = ['team', 'skills', 'privacy', 'software', 'communication']
lawyer_doc_idx = 1

print(f"\nDocumento: {document_names[lawyer_doc_idx]}")
print(f"Numero totale di documenti (N): {N}")
print(f"\nCalcoli per parole selezionate:")
print("-"*60)

for word in key_words:
    word_idx = word_to_idx[word]
    tf = word_count_matrix[lawyer_doc_idx, word_idx]
    df = df_dict[word]
    idf = math.log(N / df)
    tfidf = tf * idf
    
    print(f"\nParola: '{word}'")
    print(f"  TF (occorrenze nel doc): {tf}")
    print(f"  DF (documenti che la contengono): {df}")
    print(f"  IDF = log({N}/{df}) = log({N/df:.2f}) = {idf:.4f}")
    print(f"  TF-IDF = {tf} × {idf:.4f} = {tfidf:.4f}")

In [ ]:
# Calcolare la matrice TF-IDF completa manualmente
tfidf_matrix = np.zeros((num_docs, vocab_size))

for word_idx, word in enumerate(vocabulary_sorted):
    df = df_dict[word]
    idf = math.log(N / df)
    for doc_idx in range(num_docs):
        tf = word_count_matrix[doc_idx, word_idx]
        tfidf_matrix[doc_idx, word_idx] = tf * idf

# Creare un DataFrame per visualizzazione
df_tfidf = pd.DataFrame(
    tfidf_matrix,
    index=document_names,
    columns=vocabulary_sorted
)

print("\nMatrice TF-IDF (parole selezionate):")
print(df_tfidf[selected_cols_sklearn])

## 3.7 TF-IDF con scikit-learn

Scikit-learn fornisce la classe `TfidfVectorizer` che calcola TF-IDF in modo efficiente. Confronteremo il risultato con il nostro calcolo manuale.

In [ ]:
# Usare sklearn TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r'[a-z0-9]+', norm='l2')
X_tfidf_sklearn = tfidf_vectorizer.fit_transform(documents)
df_tfidf_sklearn = pd.DataFrame(
    X_tfidf_sklearn.toarray(),
    index=document_names,
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF con sklearn (parole selezionate):")
selected_cols_sklearn = [w for w in selected_words if w in tfidf_vectorizer.get_feature_names_out()]
print(df_tfidf_sklearn[selected_cols_sklearn])

print("\nNota: sklearn normalizza i vettori TF-IDF con L2 normalization (norma euclidea).")

In [ ]:
# Confronto tra il calcolo manuale e sklearn
print("Confronto tra calcolo manuale e sklearn:")
print("="*60)
print("\nNota: sklearn normalizza i vettori usando la norma L2.")
print("Per confrontare, normalizziamo anche il nostro calcolo manuale:")
print()

# Normalizzare il nostro calcolo manuale con L2
tfidf_matrix_normalized = np.zeros_like(tfidf_matrix)
for doc_idx in range(num_docs):
    norm = np.linalg.norm(tfidf_matrix[doc_idx, :])
    if norm > 0:
        tfidf_matrix_normalized[doc_idx, :] = tfidf_matrix[doc_idx, :] / norm

df_tfidf_normalized = pd.DataFrame(
    tfidf_matrix_normalized,
    index=document_names,
    columns=vocabulary_sorted
)

print("\nTF-IDF manuale (normalizzato con L2):")
print(df_tfidf_normalized[selected_cols_sklearn])
print("\nTF-IDF sklearn (con L2 normalization):")
print(df_tfidf_sklearn[selected_cols_sklearn])
print("\nI risultati coincidono (a meno di arrotondamenti)!")

## 3.8 One-Hot Encoding e Similarità

Un approccio ancora più semplice è il **one-hot encoding**, dove ogni parola è rappresentata come un vettore con un 1 in una posizione e 0 ovunque.

Ad esempio, se il vocabolario è: [software, team, communication, skills, privacy, ...]
- software = [1, 0, 0, 0, 0, ...]
- team = [0, 1, 0, 0, 0, ...]
- communication = [0, 0, 1, 0, 0, ...]

**Problema**: In questo schema, tutte le parole sono ugualmente distanti! La similarità coseno tra due parole diverse qualsiasi è sempre 0. Questo non cattura alcuna relazione semantica tra le parole.

In [ ]:
# Dimostrare il problema del one-hot encoding
print("Problema del One-Hot Encoding:")
print("="*60)

# Selezionare alcune parole per la dimostrazione
words_for_demo = ['software', 'team', 'communication', 'skills', 'privacy']
word_indices = [word_to_idx[w] for w in words_for_demo]

# Creare vettori one-hot
one_hot_vectors = {}
for word in words_for_demo:
    vec = np.zeros(len(vocabulary_sorted))
    vec[word_to_idx[word]] = 1
    one_hot_vectors[word] = vec

print(f"\nVocabolario: {vocabulary_sorted[:10]}...")
print(f"\nVettori one-hot per le parole selezionate:")
for word in words_for_demo:
    vec = one_hot_vectors[word]
    nonzero_idx = np.where(vec != 0)[0][0]
    print(f"  '{word}': posizione {nonzero_idx}, vettore = [0, 0, ..., 1, ..., 0]")

print(f"\n\nSimilarità coseno tra coppie di parole (one-hot):")
for i, word1 in enumerate(words_for_demo):
    for word2 in words_for_demo[i+1:]:
        sim = cosine_similarity([one_hot_vectors[word1]], [one_hot_vectors[word2]])[0, 0]
        print(f"  '{word1}' vs '{word2}': {sim:.4f}")

print("\n⚠️ PROBLEMA: Tutte le similitudini sono 0!")
print("Il one-hot encoding non cattura relazioni semantiche tra le parole.")
print("Per questo motivo, gli word embeddings (es. Word2Vec, GloVe) sono superiori.")

## 3.9 Similarità tra documenti con BoW/TF-IDF

Nonostante il problema del one-hot encoding per le singole parole, possiamo comunque usare BoW e TF-IDF per misurare la similarità tra documenti completi usando la similarità coseno.

In [ ]:
# Calcolare la matrice di similarità coseno usando Word Count (BoW)
print("Similarità coseno tra documenti (usando Word Count):")
print("="*60)

similarity_bow = cosine_similarity(word_count_matrix)
similarity_bow_df = pd.DataFrame(
    similarity_bow,
    index=document_names,
    columns=document_names
)

print("\nMatrice di similarità (Word Count):")
print(similarity_bow_df.round(4))
print()

# Trovare le coppie più simili
print("Coppie più simili (escluso il documento con se stesso):")
for i in range(len(document_names)):
    for j in range(i+1, len(document_names)):
        sim = similarity_bow[i, j]
        print(f"  {document_names[i]} ↔ {document_names[j]}: {sim:.4f}")

In [ ]:
# Calcolare la matrice di similarità coseno usando TF-IDF
print("\nSimilarità coseno tra documenti (usando TF-IDF normalizzato):")
print("="*60)

similarity_tfidf = cosine_similarity(tfidf_matrix_normalized)
similarity_tfidf_df = pd.DataFrame(
    similarity_tfidf,
    index=document_names,
    columns=document_names
)

print("\nMatrice di similarità (TF-IDF):")
print(similarity_tfidf_df.round(4))
print()

print("Coppie più simili (escluso il documento con se stesso):")
for i in range(len(document_names)):
    for j in range(i+1, len(document_names)):
        sim = similarity_tfidf[i, j]
        print(f"  {document_names[i]} ↔ {document_names[j]}: {sim:.4f}")

print("\n📊 Osservazioni:")
print("- Con TF-IDF, i documenti risultano meno simili rispetto a BoW")
print("- Questo perché TF-IDF riduce il peso di parole comuni come 'team' e 'collaborate'")
print("- Le similarità più alte riflettono il contenuto specifico di ogni documento")

In [ ]:
# Usare direttamente sklearn TfidfVectorizer per le similarità
print("\nVerifica con sklearn TfidfVectorizer:")
print("="*60)

similarity_tfidf_sklearn = cosine_similarity(X_tfidf_sklearn)
similarity_tfidf_sklearn_df = pd.DataFrame(
    similarity_tfidf_sklearn,
    index=document_names,
    columns=document_names
)

print("\nMatrice di similarità (sklearn TfidfVectorizer):")
print(similarity_tfidf_sklearn_df.round(4))

# ESERCIZIO

## Consegna

In questa esercizio, dovete lavorare con 5 NUOVE offerte di lavoro:

1. **Data Scientist**: "We are seeking a talented Data Scientist to join our analytics team. Analyze large datasets and develop machine learning models. Collaborate with engineers to deploy your models in production. Your data science expertise will drive business decisions."

2. **Marketing Manager**: "Join our creative Marketing team. Develop marketing strategies to reach target audiences. Collaborate with sales and design teams. Your marketing skills and creative vision will shape our brand narrative."

3. **UX Designer**: "We need an experienced UX Designer. Create user-centered designs and conduct user research. Collaborate with development teams to implement your designs. Your design skills will enhance user experience and satisfaction."

4. **Financial Analyst**: "Exciting Financial Analyst position. Analyze financial data and create reports. Collaborate with accounting and finance teams. Your analytical skills will support financial planning and compliance."

5. **HR Specialist**: "Join our Human Resources team. Develop HR strategies and manage recruitment. Collaborate with department managers. Your HR skills will support employee growth and organizational success."

## Compiti da svolgere:

(a) **Costruire il vocabolario manualmente**: estrai tutte le parole univoche dal corpus preprocessato.

(b) **Creare la matrice Word Presence**: rappresenta il corpus come una matrice 5×V (5 documenti, V parole del vocabolario) con valori 0/1.

(c) **Creare la matrice Word Count**: rappresenta il corpus come una matrice 5×V con il numero di occorrenze di ogni parola.

(d) **Calcolare TF-IDF manualmente**: per almeno 3 parole selezionate nel documento "Data Scientist", calcola i valori TF-IDF mostrando i passaggi intermedi.

(e) **Calcolare TF-IDF con sklearn**: usa TfidfVectorizer per calcolare la matrice TF-IDF per tutto il corpus.

(f) **Trovare documenti più simili**: calcola la matrice di similarità coseno usando TF-IDF e identifica quale coppia di documenti è più simile.

## Spazio per la soluzione

**Inserisci il tuo codice qui di seguito:**

In [ ]:
# Inserire la soluzione qui
# (a) Costruire il vocabolario manualmente








In [ ]:
# (b) Creare la matrice Word Presence






In [ ]:
# (c) Creare la matrice Word Count






In [ ]:
# (d) Calcolare TF-IDF manualmente






In [ ]:
# (e) Calcolare TF-IDF con sklearn






In [ ]:
# (f) Trovare documenti più simili






# SOLUZIONE

Di seguito è riportata la soluzione completa dell'esercizio con tutti i passaggi dettagliati.

In [ ]:
# Definire i 5 documenti per l'esercizio
exercise_docs = [
    "We are seeking a talented Data Scientist to join our analytics team. Analyze large datasets and develop machine learning models. Collaborate with engineers to deploy your models in production. Your data science expertise will drive business decisions.",  # Data Scientist
    "Join our creative Marketing team. Develop marketing strategies to reach target audiences. Collaborate with sales and design teams. Your marketing skills and creative vision will shape our brand narrative.",  # Marketing Manager
    "We need an experienced UX Designer. Create user-centered designs and conduct user research. Collaborate with development teams to implement your designs. Your design skills will enhance user experience and satisfaction.",  # UX Designer
    "Exciting Financial Analyst position. Analyze financial data and create reports. Collaborate with accounting and finance teams. Your analytical skills will support financial planning and compliance.",  # Financial Analyst
    "Join our Human Resources team. Develop HR strategies and manage recruitment. Collaborate with department managers. Your HR skills will support employee growth and organizational success."  # HR Specialist
]

exercise_doc_names = ["Data Scientist", "Marketing Manager", "UX Designer", "Financial Analyst", "HR Specialist"]

print("\n" + "="*70)
print("SOLUZIONE ESERCIZIO - Bag of Words e TF-IDF")
print("="*70)
print("\n5 nuove offerte di lavoro:")
for i, (name, doc) in enumerate(zip(exercise_doc_names, exercise_docs), 1):
    print(f"\n{i}. {name}:")
    print(f"   {doc[:75]}...")

In [ ]:
# PARTE (a): Costruire il vocabolario manualmente
print("\n\n" + "="*70)
print("(a) COSTRUIRE IL VOCABOLARIO MANUALMENTE")
print("="*70)

# Preprocessare i documenti
exercise_preprocessed = [preprocess_text(doc) for doc in exercise_docs]

# Costruire il vocabolario
exercise_vocab = set()
for tokens in exercise_preprocessed:
    exercise_vocab.update(tokens)

exercise_vocab_sorted = sorted(list(exercise_vocab))
exercise_word_to_idx = {word: idx for idx, word in enumerate(exercise_vocab_sorted)}
exercise_idx_to_word = {idx: word for word, idx in exercise_word_to_idx.items()}

print(f"\nDimensione del vocabolario: {len(exercise_vocab_sorted)} parole")
print(f"\nVocabolario completo (in ordine alfabetico):")
for i, word in enumerate(exercise_vocab_sorted):
    print(f"  {i:2d}: {word}")
    if (i + 1) % 20 == 0:
        print()

In [ ]:
# PARTE (b): Creare la matrice Word Presence
print("\n\n" + "="*70)
print("(b) MATRICE WORD PRESENCE (0/1)")
print("="*70)

exercise_num_docs = len(exercise_docs)
exercise_vocab_size = len(exercise_vocab_sorted)

exercise_word_presence = np.zeros((exercise_num_docs, exercise_vocab_size))

for doc_idx, tokens in enumerate(exercise_preprocessed):
    unique_words = set(tokens)
    for word in unique_words:
        word_idx = exercise_word_to_idx[word]
        exercise_word_presence[doc_idx, word_idx] = 1

exercise_df_presence = pd.DataFrame(
    exercise_word_presence,
    index=exercise_doc_names,
    columns=exercise_vocab_sorted
)

print(f"\nShape della matrice: {exercise_df_presence.shape}")
print(f"(5 documenti × {exercise_vocab_size} parole)")
print(f"\nMatrice Word Presence (prime 20 parole):")
print(exercise_df_presence.iloc[:, :20])
print(f"\n... (altre {exercise_vocab_size - 20} parole)")

In [ ]:
# PARTE (c): Creare la matrice Word Count
print("\n\n" + "="*70)
print("(c) MATRICE WORD COUNT (frequenze)")
print("="*70)

exercise_word_count = np.zeros((exercise_num_docs, exercise_vocab_size))

for doc_idx, tokens in enumerate(exercise_preprocessed):
    word_freq = Counter(tokens)
    for word, count in word_freq.items():
        word_idx = exercise_word_to_idx[word]
        exercise_word_count[doc_idx, word_idx] = count

exercise_df_count = pd.DataFrame(
    exercise_word_count,
    index=exercise_doc_names,
    columns=exercise_vocab_sorted
)

print(f"\nMatrice Word Count (prime 20 parole):")
print(exercise_df_count.iloc[:, :20])
print(f"\n... (altre {exercise_vocab_size - 20} parole)")

print(f"\n\nStatistiche Word Count:")
for doc_idx, doc_name in enumerate(exercise_doc_names):
    total_words = exercise_df_count.iloc[doc_idx].sum()
    unique_words = (exercise_df_count.iloc[doc_idx] > 0).sum()
    print(f"  {doc_name}: {int(total_words)} parole totali, {unique_words} parole univoche")

In [ ]:
# PARTE (d): Calcolare TF-IDF manualmente
print("\n\n" + "="*70)
print("(d) CALCOLO TF-IDF MANUALE (documento 'Data Scientist')")
print("="*70)

# Calcolare document frequency
exercise_df_dict = {}
for word_idx, word in enumerate(exercise_vocab_sorted):
    count = 0
    for doc_idx in range(exercise_num_docs):
        if exercise_word_count[doc_idx, word_idx] > 0:
            count += 1
    exercise_df_dict[word] = count

# Selezionare parole chiave per il documento Data Scientist
N_exercise = exercise_num_docs
data_scientist_idx = 0

key_words_exercise = ['scientist', 'data', 'analyze', 'team', 'collaborate', 'models', 'learning']
key_words_exercise = [w for w in key_words_exercise if w in exercise_vocab_sorted]

print(f"\nDocumento: 'Data Scientist'")
print(f"Numero totale di documenti (N): {N_exercise}")
print(f"\nCalcoli TF-IDF per parole selezionate:")
print("-" * 70)

manual_tfidf_values = {}
for word in key_words_exercise:
    word_idx = exercise_word_to_idx[word]
    tf = exercise_word_count[data_scientist_idx, word_idx]
    df = exercise_df_dict[word]
    idf = math.log(N_exercise / df) if df > 0 else 0
    tfidf = tf * idf
    manual_tfidf_values[word] = tfidf
    
    print(f"\nParola: '{word}'")
    print(f"  TF (occorrenze nel doc Data Scientist): {int(tf)}")
    print(f"  DF (numero di documenti che la contengono): {df}")
    print(f"  IDF = log({N_exercise}/{df}) = log({N_exercise/df:.4f}) = {idf:.6f}")
    print(f"  TF-IDF = {int(tf)} × {idf:.6f} = {tfidf:.6f}")

In [ ]:
# Calcolare la matrice TF-IDF completa manualmente
print("\n\n" + "="*70)
print("Matrice TF-IDF completa (calcolo manuale):")
print("="*70)

exercise_tfidf_matrix = np.zeros((exercise_num_docs, exercise_vocab_size))

for word_idx, word in enumerate(exercise_vocab_sorted):
    df = exercise_df_dict[word]
    idf = math.log(N_exercise / df) if df > 0 else 0
    for doc_idx in range(exercise_num_docs):
        tf = exercise_word_count[doc_idx, word_idx]
        exercise_tfidf_matrix[doc_idx, word_idx] = tf * idf

exercise_df_tfidf = pd.DataFrame(
    exercise_tfidf_matrix,
    index=exercise_doc_names,
    columns=exercise_vocab_sorted
)

print(f"\nMatrice TF-IDF (prime 20 parole):")
print(exercise_df_tfidf.iloc[:, :20].round(6))
print(f"\n... (altre {exercise_vocab_size - 20} parole)")

In [ ]:
# PARTE (e): Calcolare TF-IDF con sklearn
print("\n\n" + "="*70)
print("(e) TF-IDF CON SCIKIT-LEARN (TfidfVectorizer)")
print("="*70)

# Usare sklearn TfidfVectorizer
exercise_tfidf_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r'[a-z0-9]+', norm='l2')
exercise_X_tfidf_sklearn = exercise_tfidf_vectorizer.fit_transform(exercise_docs)
exercise_df_tfidf_sklearn = pd.DataFrame(
    exercise_X_tfidf_sklearn.toarray(),
    index=exercise_doc_names,
    columns=exercise_tfidf_vectorizer.get_feature_names_out()
)

print(f"\nShape della matrice TF-IDF da sklearn: {exercise_df_tfidf_sklearn.shape}")
print(f"\nMatrice TF-IDF con sklearn (prime 20 parole):")
print(exercise_df_tfidf_sklearn.iloc[:, :20].round(6))
print(f"\n... (altre {exercise_df_tfidf_sklearn.shape[1] - 20} parole)")

print("\n✓ Nota: sklearn normalizza i vettori con L2 normalization.")

In [ ]:
# PARTE (f): Trovare documenti più simili
print("\n\n" + "="*70)
print("(f) TROVARE I DOCUMENTI PIÙ SIMILI")
print("="*70)

# Calcolare la matrice di similarità coseno usando TF-IDF di sklearn
exercise_similarity_tfidf = cosine_similarity(exercise_X_tfidf_sklearn)
exercise_similarity_tfidf_df = pd.DataFrame(
    exercise_similarity_tfidf,
    index=exercise_doc_names,
    columns=exercise_doc_names
)

print("\nMatrice di similarità coseno (TF-IDF):")
print(exercise_similarity_tfidf_df.round(6))

print("\n\nSimilarità tra coppie di documenti (escluso il documento con se stesso):")
print("-" * 70)

similarity_pairs = []
for i in range(len(exercise_doc_names)):
    for j in range(i+1, len(exercise_doc_names)):
        sim = exercise_similarity_tfidf[i, j]
        similarity_pairs.append((exercise_doc_names[i], exercise_doc_names[j], sim))
        print(f"  {exercise_doc_names[i]:20s} ↔ {exercise_doc_names[j]:20s}: {sim:.6f}")

# Trovare la coppia più simile
most_similar_pair = max(similarity_pairs, key=lambda x: x[2])
print(f"\n\n🎯 COPPIA PIÙ SIMILE:")
print(f"   '{most_similar_pair[0]}' ↔ '{most_similar_pair[1]}'")
print(f"   Similarità: {most_similar_pair[2]:.6f}")

In [ ]:
# Analisi delle similarità
print("\n\n" + "="*70)
print("ANALISI DELLE SIMILARITÀ")
print("="*70)

print("\nOsservazioni principali:")
print("-" * 70)

# Ordinare le coppie per similarità
similarity_pairs_sorted = sorted(similarity_pairs, key=lambda x: x[2], reverse=True)

print("\nRanking delle coppie più simili:")
for idx, (doc1, doc2, sim) in enumerate(similarity_pairs_sorted, 1):
    print(f"  {idx}. {doc1:20s} ↔ {doc2:20s}: {sim:.6f}")

print("\n\nParole chiave per ogni documento:")
for doc_idx, doc_name in enumerate(exercise_doc_names):
    # Trovare le 5 parole con il più alto TF-IDF
    tfidf_row = exercise_df_tfidf_sklearn.iloc[doc_idx]
    top_words = tfidf_row.nlargest(5)
    print(f"\n  {doc_name}:")
    for word, tfidf_val in top_words.items():
        print(f"    - {word}: {tfidf_val:.6f}")

In [ ]:
# Riepilogo della soluzione
print("\n\n" + "="*70)
print("RIEPILOGO DELLA SOLUZIONE")
print("="*70)

print(f"""
✓ (a) Vocabolario costruito: {len(exercise_vocab_sorted)} parole univoche

✓ (b) Matrice Word Presence: {exercise_word_presence.shape[0]} documenti × {exercise_word_presence.shape[1]} parole
    Rappresentazione binaria (0/1) della presenza di parole
    
✓ (c) Matrice Word Count: {exercise_word_count.shape[0]} documenti × {exercise_word_count.shape[1]} parole
    Numero di occorrenze di ogni parola in ogni documento
    
✓ (d) TF-IDF manuale: Calcolati per {len(key_words_exercise)} parole nel documento 'Data Scientist'
    Formula: TF-IDF = TF × log(N/DF)
    
✓ (e) TF-IDF con sklearn: Matrice {exercise_df_tfidf_sklearn.shape[0]} × {exercise_df_tfidf_sklearn.shape[1]}
    Vettori normalizzati con L2 normalization
    
✓ (f) Similarità tra documenti:
    Documento 1: {most_similar_pair[0]}
    Documento 2: {most_similar_pair[1]}
    Similarità coseno: {most_similar_pair[2]:.6f}
    
    Questo indica che questi due documenti hanno il contenuto più simile
    tra tutte le coppie di offerte di lavoro.
""")

print("="*70)